# Credit Default Risk Scorecard
## Notebook 2: Feature Engineering
**Author:** Simpson Gundlapally

Clean the dataset and create new features that improve model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(f'Raw data: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Step 1 — Handle Missing Values

In [ ]:
print('Missing values before cleaning:')
print(df.isnull().sum())

# MonthlyIncome — fill with median (robust to outliers)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

# NumberOfDependents — fill with 0 (safest assumption)
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

print('\nMissing values after cleaning:')
print(df.isnull().sum())

## Step 2 — Remove Invalid Records

In [ ]:
print(f'Before: {len(df):,} rows')

# Remove age < 18 (invalid)
df = df[df['age'] >= 18].copy()

# Cap outliers — utilisation must be 0-1
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)

# Cap extreme debt ratios
df['DebtRatio'] = df['DebtRatio'].clip(0, 10)

print(f'After:  {len(df):,} rows')

## Step 3 — Create New Features

In [ ]:
# Age bands for analysis
df['AgeBand'] = pd.cut(df['age'],
    bins=[17,25,35,45,55,65,120],
    labels=['18-25','26-35','36-45','46-55','56-65','65+'])

# Utilisation bands
df['UtilBand'] = pd.cut(df['RevolvingUtilizationOfUnsecuredLines'],
    bins=[-0.001,0.3,0.6,0.9,1.001],
    labels=['Low','Medium','High','Very High'])

# Total late payments across all categories
df['TotalLatePays'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# Debt exposure in absolute terms
df['DebtToIncomeRatio'] = df['DebtRatio'] * df['MonthlyIncome']

# How densely has this person used credit relative to age
df['CreditDensity'] = df['NumberOfOpenCreditLinesAndLoans'] / (df['age'] - 17 + 0.1)

print('New features created:')
print(['AgeBand','UtilBand','TotalLatePays','DebtToIncomeRatio','CreditDensity'])
print(f'\nDataset now has {df.shape[1]} columns')

## Step 4 — Validate Features

In [ ]:
# Default rate by TotalLatePays
print('Default rate by total late payments:')
print(df.groupby('TotalLatePays')['SeriousDlqin2yrs'].agg(['mean','count']).head(10).round(4))

# Save cleaned data
df.to_csv('../data/cs-training-cleaned.csv', index=False)
print('\nCleaned dataset saved to data/cs-training-cleaned.csv')